In [2]:
import numpy as np
from src.analysis.processing import lime_ranking, shap_ranking, shapiq_ranking, meg_ranking, mmace_ranking, \
    meg_cf_percent, mmace_cf_percent
from src.analysis.xai_eval import pgi, pgu
import pickle
import os
import joblib
from joblib import Parallel, delayed

max_k = 5
dataset_name = 'qm9_simple_linear6'  # Change this to the dataset you want to analyze
results_dir = f'../results/synthetic_data/{dataset_name}/explanations'
model_dir = f'../results/synthetic_data/{dataset_name}/'

target = 'target'

results_dict = {
    'lime': ('lime_results.pickle', lime_ranking),
    'shap': ('shap_results.pickle', shap_ranking),
    'shapiq1': ('shapiq1_results.pickle', shapiq_ranking),
    'shapiq2': ('shapiq2_results.pickle', shapiq_ranking),
    'meg': ('meg2_results.pickle', meg_ranking, meg_cf_percent),
    'mmace': ('mmace_results.pickle', mmace_ranking, mmace_cf_percent),
}

ranking_dict = {}
ranking_per_fold_dict = {}
cf_similarity_dict = {}
cf_validity_dict = {}
metrics_dict = {}

def run(key):
    print(key)
    file_name, ranking_func = results_dict[key][:2]

    if len(results_dict[key]) > 2:
        cf_func = results_dict[key][2]
    else:
        cf_func = None
    with open(os.path.join(results_dir, file_name), 'rb') as f:
        results = pickle.load(f)

    ranking, rankings_per_fold = ranking_func(results, target)
    if cf_func is not None:
        cf_percent = cf_func(results, target=target)
    else:
        cf_percent = None

    if cf_percent is not None:
        print(f"{key} counterfactual percent: {cf_percent}")

    pgis, pgus = [], []
    pgis_org, pgus_org = [], []

    ranking_dict[key] = ranking
    ranking_per_fold_dict[key] = rankings_per_fold
    if cf_percent is not None:
        cf_validity_dict[key] = cf_percent[0]
        cf_similarity_dict[key] = cf_percent[1]

    for i in range(len(rankings_per_fold)):
        print(i)
        model = os.path.join(model_dir, f'model_{i}.joblib')
        model = joblib.load(model)
        test_examples = results['test_data'][i].drop(columns=[target])
        train_examples = results['training_data'][i].drop(columns=[target])

        ranking_current = list(rankings_per_fold[i]['features'])

        pgi_one, pgi_org = pgi(test_examples, ranking_current, model, train_examples)
        pgu_one, pgu_org = pgu(test_examples, ranking_current, model, train_examples)

        pgis.append(pgi_one)
        pgus.append(pgu_one)

        pgis_org.append(pgi_org)
        pgus_org.append(pgu_org)

    pgi_mean = np.mean(pgis)
    pgu_mean = np.mean(pgus)
    pgi_std = np.std(pgis)
    pgu_std = np.std(pgus)
    pgi_org_mean = np.mean(pgis_org)
    pgu_org_mean = np.mean(pgus_org)
    pgi_org_std = np.std(pgis_org)
    pgu_org_std = np.std(pgus_org)

    print(f"{key} PGI: {pgi_mean} ({pgi_std}), PGU: {pgu_mean} ({pgu_std})")
    print(f"{key} PGI Org: {pgi_org_mean} ({pgi_org_std}), PGU Org: {pgu_org_mean} ({pgu_org_std})")
    return key, {
        'pgi_mean': pgi_mean,
        'pgu_mean': pgu_mean,
        'pgi_std': pgi_std,
        'pgu_std': pgu_std,
        'pgi_org_mean': pgi_org_mean,
        'pgu_org_mean': pgu_org_mean,
        'pgi_org_std': pgi_org_std,
        'pgu_org_std': pgu_org_std,
    }
metrics_dict = Parallel(n_jobs=5)(delayed(run)(key) for key in results_dict.keys())
metrics_dict = {key: value for key, value in metrics_dict}

lime
0
meg
shapiq1
shapiq2
shap
0
0
meg counterfactual percent: (np.float64(0.8846), np.float64(0.31668162317903115), np.float64(0.09789134043137054))
0
0


KeyboardInterrupt: 

In [2]:
import pandas as pd
def convert_term_ranking_to_feature_ranking(ranking_with_interactions: list) -> list:
    """
    Converts a ranking of terms (features + interactions) into a ranking of
    only features based on their earliest appearance.
    """
    feature_to_best_rank = {}
    for i, term in enumerate(ranking_with_interactions):
        constituent_features = term.split(' x ')
        for feature in constituent_features:
            if feature not in feature_to_best_rank:
                feature_to_best_rank[feature] = i

    # Sort the features by their best rank
    sorted_features = sorted(feature_to_best_rank.items(), key=lambda item: item[1])

    return [feature for feature, rank in sorted_features]

def aggregate_rankings_by_mean_position(list_of_rankings: list) -> list:
    if not list_of_rankings:
        return []
    all_items = set()
    for ranking in list_of_rankings:
        all_items.update(ranking)
    item_scores = {}
    for item in all_items:
        positions = []
        for ranking in list_of_rankings:
            try:
                position = ranking.index(item)
            except ValueError:
                position = len(ranking)
            positions.append(position)
        item_scores[item] = np.mean(positions)
    sorted_items = sorted(item_scores.keys(), key=lambda item: item_scores[item])
    return sorted_items

In [9]:
pgis, pgus = [], []
pgis_org, pgus_org = [], []
for i in range(len(ranking_per_fold_dict['lime'])):

    model = os.path.join(model_dir, f'model_{i}.joblib')
    model = joblib.load(model)
    test_examples = results['test_data'][i].drop(columns=[target])
    train_examples = results['training_data'][i].drop(columns=[target])

    rankings = []
    for key in ranking_per_fold_dict.keys():
        ranking_current = list(ranking_per_fold_dict[key][i]['features'])
        if key == 'shapiq2':
            ranking_current = convert_term_ranking_to_feature_ranking(ranking_current)
        rankings.append(ranking_current)

    aggregated_ranking = aggregate_rankings_by_mean_position(rankings)
    pgi_one, pgi_org = pgi(test_examples, aggregated_ranking, model, train_examples)
    pgu_one, pgu_org = pgu(test_examples, aggregated_ranking, model, train_examples)

    pgis.append(pgi_one)
    pgus.append(pgu_one)

    pgis_org.append(pgi_org)
    pgus_org.append(pgu_org)

pgi_mean = np.mean(pgis)
pgu_mean = np.mean(pgus)
pgi_std = np.std(pgis)
pgu_std = np.std(pgus)
pgi_org_mean = np.mean(pgis_org)
pgu_org_mean = np.mean(pgus_org)
pgi_org_std = np.std(pgis_org)
pgu_org_std = np.std(pgus_org)

metrics_dict['aggregated'] = {
        'pgi_mean': pgi_mean,
        'pgu_mean': pgu_mean,
        'pgi_std': pgi_std,
        'pgu_std': pgu_std,
        'pgi_org_mean': pgi_org_mean,
        'pgu_org_mean': pgu_org_mean,
        'pgi_org_std': pgi_org_std,
        'pgu_org_std': pgu_org_std,
    }

print(f"Aggregated PGI: {pgi_mean} ({pgi_std}), PGU: {pgu_mean} ({pgu_std})")
print(f"Aggregated PGI Org: {pgi_org_mean} ({pgi_org_std}), PGU Org: {pgu_org_mean} ({pgu_org_std})")

Aggregated PGI: 0.517119711807953 (0.08889054757494588), PGU: 0.061903698142985894 (0.013862959242836993)
Aggregated PGI Org: 13.221888814631566 (0.9413014834417209), PGU Org: 1.5777085529002428 (0.25649393311477703)
Aggregated PGI 10: 0.5291890357738442 (0.114587760712076), PGU 10: 0.04315744505876437 (0.011221208698040434)
Aggregated PGI Org 10: 13.423009216839919 (1.003177237951579), PGU Org 10: 1.0941251466307766 (0.2055294029532347)


In [10]:
# Save the results
os.makedirs(os.path.join(results_dir, 'analysis'), exist_ok=True)
with open(os.path.join(results_dir, 'analysis', 'metrics_results.pickle'), 'wb') as f:
    pickle.dump(metrics_dict, f)
with open(os.path.join(results_dir, 'analysis', 'ranking_results.pickle'), 'wb') as f:
    pickle.dump(ranking_dict, f)
with open(os.path.join(results_dir, 'analysis', 'ranking_per_fold_results.pickle'), 'wb') as f:
    pickle.dump(ranking_per_fold_dict, f)
with open(os.path.join(results_dir, 'analysis', 'cf_validity_results.pickle'), 'wb') as f:
    pickle.dump(cf_validity_dict, f)
with open(os.path.join(results_dir, 'analysis', 'cf_similarity_results.pickle'), 'wb') as f:
    pickle.dump(cf_similarity_dict, f)

In [23]:
from src.analysis.xai_eval import rank_correlation

def convert_term_ranking_to_feature_ranking(ranking_with_interactions: pd.DataFrame) -> list:
    """
    Converts a ranking of terms (features + interactions) into a ranking of
    only features based on their earliest appearance.
    """
    ranking_with_interactions['rank'] = ranking_with_interactions['abs_ranking'].rank('min', ascending=False).astype(int)
    feature_to_best_rank = {}
    for i in range(len(ranking_with_interactions)):
        term = ranking_with_interactions.iloc[i]['features']
        idx = ranking_with_interactions.iloc[i]['rank']
        constituent_features = term.split(' x ')
        for feature in constituent_features:
            if feature not in feature_to_best_rank:
                feature_to_best_rank[feature] = idx
            else:
                feature_to_best_rank[feature] = min(feature_to_best_rank[feature], idx)

    return feature_to_best_rank

#calculate rankings correlations
pairs_of_ranks = [(key1, key2) for key1 in ranking_per_fold_dict.keys() for key2 in ranking_per_fold_dict.keys()]
correlations = {}
for key1, key2 in pairs_of_ranks:
    corrs = []
    for i in range(len(ranking_per_fold_dict[key1])):
        rank1 = ranking_per_fold_dict[key1][i]
        rank2 = ranking_per_fold_dict[key2][i]
        if key1 == 'shapiq2':
            rank1 = convert_term_ranking_to_feature_ranking(rank1)
        if key2 == 'shapiq2':
            rank2 = convert_term_ranking_to_feature_ranking(rank2)
        if not isinstance(rank1, dict):
            rank1['rank'] = rank1['abs_ranking'].rank('min', ascending=False).astype(int)
            rank1 = {f: r for f, r in zip(rank1['features'], rank1['rank'])}
        if not isinstance(rank2, dict):
            rank2['rank'] = rank2['abs_ranking'].rank('min', ascending=False).astype(int)
            rank2 = {f: r for f, r in zip(rank2['features'], rank2['rank'])}
        c = rank_correlation(rank1, rank2).statistic
        corrs.append(c)
    correlation = np.mean(corrs)
    correlations[(key1, key2)] = correlation
    print(f"{key1} vs {key2}: {correlation:.4f}")
    print('--' * 20)

with open(os.path.join(results_dir, 'analysis', 'correlations_results.pickle'), 'wb') as f:
    pickle.dump(correlations, f)

lime vs lime: 1.0000
----------------------------------------
lime vs shap: 0.3713
----------------------------------------
lime vs shapiq1: 0.3598
----------------------------------------
lime vs shapiq2: 0.3708
----------------------------------------
lime vs meg: 0.1787
----------------------------------------
lime vs mmace: 0.2568
----------------------------------------
shap vs lime: 0.3713
----------------------------------------
shap vs shap: 1.0000
----------------------------------------
shap vs shapiq1: 0.3902
----------------------------------------
shap vs shapiq2: 0.4688
----------------------------------------
shap vs meg: 0.4072
----------------------------------------
shap vs mmace: 0.4063
----------------------------------------
shapiq1 vs lime: 0.3598
----------------------------------------
shapiq1 vs shap: 0.3902
----------------------------------------
shapiq1 vs shapiq1: 1.0000
----------------------------------------
shapiq1 vs shapiq2: 0.3980
-------------------